# Projeto A2 — Rota de visitas com Solomon C102

**Equipe:** Guilherme Caleffi, João Franco e Marcio Henrique  
**Trilha 5:** Caixeiro Viajante (TSP), vizinho mais próximo e 2-Opt.

O depósito representa o hotel. Visitamos os clientes selecionados e retornamos à base, minimizando distância euclidiana. Usamos um benchmark público adaptado ao cenário comercial.

Dataset obtido no [Kaggle — Solomon benchmark](https://www.kaggle.com/datasets/jaikeerthy/solomon-benchmark). Consulte docs/ENUNCIADO.md e docs/MODELAGEM-FORMAL.md.

## Preparação
Localmente, abra este notebook na raiz do projeto e instale requirements.txt no ambiente do kernel. No Colab, envie e extraia o ZIP do repositório, mantendo a estrutura de pastas; ajuste RAIZ_PROJETO abaixo para a pasta extraída. Instale matplotlib no kernel se necessário, com %pip install matplotlib.


In [ ]:
from pathlib import Path
import sys

RAIZ_PROJETO = Path.cwd()  # Colab: ajuste para a pasta extraída do projeto
necessarios = ['dados_io.py', 'roteiro_tsp.py', 'executar_roteiro.py',
               'dados/solomon/C102_paradas.csv']
faltantes = [nome for nome in necessarios if not (RAIZ_PROJETO / nome).is_file()]
if faltantes:
    raise FileNotFoundError(f'Ajuste RAIZ_PROJETO. Arquivos ausentes: {faltantes}')
sys.path.insert(0, str(RAIZ_PROJETO.resolve()))

from dados_io import carregar_paradas, deposito, paradas_por_ids
from roteiro_tsp import matriz_distancias, ordem_ingenua, vizinho_mais_proximo
from roteiro_tsp import otimizar_2opt_fechado, custo_rota_fechada
from executar_roteiro import CLIENTES_SELECIONADOS, plotar_rota

SAIDA = RAIZ_PROJETO / 'saida'
SAIDA.mkdir(exist_ok=True)


## 1. Dados e recorte
O CSV preserva o benchmark completo: 100 clientes e um depósito. Usamos cust_no para identificação e x/y para distâncias em unidades do benchmark. Horários, serviço, demanda e capacidade não restringem esta versão. O recorte padrão do script é de 15 clientes (IDs 1 a 15), facilitando a visualização.


In [ ]:
todas = carregar_paradas(RAIZ_PROJETO / 'dados/solomon/C102_paradas.csv')
ids = [p.cust_no for p in todas]
assert len(todas) == 101, 'Esperados 101 registros no C102 completo'
assert len(set(ids)) == len(ids), 'Identificadores duplicados'
assert set(ids) == set(range(101)), 'Esperados IDs de 0 a 100'
assert len(set(CLIENTES_SELECIONADOS)) == len(CLIENTES_SELECIONADOS)

base = deposito(todas)
visitas = paradas_por_ids(todas, CLIENTES_SELECIONADOS)
pontos = [base] + visitas
dist = matriz_distancias(pontos)
n = len(visitas)
print(f'Base completa: {len(todas)} registros; depósito {base.cust_no} em ({base.x}, {base.y})')
print('Clientes selecionados:', CLIENTES_SELECIONADOS)
print(f'Matriz de distâncias: {len(dist)} x {len(dist)}')


## 2. Construção e melhoria da rota
Calculamos distâncias euclidianas entre pares de pontos. O vizinho mais próximo escolhe a parada ainda não visitada mais próxima. Depois, o 2-Opt testa inversões de segmentos e aceita as que reduzem a distância. Encerra sem melhoria ou ao atingir 500 iterações; não há garantia de ótimo global.

A referência de comparação é a ordem de entrada dos clientes, crescente por ID no exemplo padrão. Os índices da matriz são locais; a saída é convertida para os IDs originais.


In [ ]:
ingenua = ordem_ingenua(n)
inicial = vizinho_mais_proximo(dist, n)
melhorada = otimizar_2opt_fechado(dist, inicial)
c_ingenua = custo_rota_fechada(dist, ingenua)
c_inicial = custo_rota_fechada(dist, inicial)
c_final = custo_rota_fechada(dist, melhorada)

for ordem in (ingenua, inicial, melhorada):
    assert sorted(ordem) == list(range(1, n + 1)), 'Cada cliente deve aparecer uma vez'
assert c_final <= c_inicial + 1e-9, 'A melhoria não pode aumentar o custo'
print(f'Ordem ingênua: {c_ingenua:.2f}')
print(f'Vizinho mais próximo: {c_inicial:.2f}')
print(f'Após 2-Opt: {c_final:.2f}')
if c_ingenua > 0:
    print(f'Redução sobre a ordem ingênua: {(1 - c_final / c_ingenua) * 100:.1f}%')
sequencia = [base.cust_no] + [pontos[i].cust_no for i in melhorada] + [base.cust_no]
print('Rota fechada:', ' -> '.join(map(str, sequencia)))


## 3. Visualização e interpretação
Os gráficos mostram a ordem de entrada e a rota após as heurísticas. O ganho representa distância no benchmark, sem conversão para quilômetros, tempo, dinheiro ou trajetos por ruas. A comparação não comprova ótimo global.


In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None  # Executado fora de um ambiente Jupyter

plotar_rota(pontos, ingenua, 'Ordem de entrada', SAIDA / 'rota_ingenua.png')
plotar_rota(pontos, melhorada, 'Rota melhorada por 2-Opt', SAIDA / 'rota_otimizada.png')
for nome in ('rota_ingenua.png', 'rota_otimizada.png'):
    if Image is not None:
        display(Image(filename=str(SAIDA / nome)))
    else:
        print(SAIDA / nome)


## 4. Execução completa
A célula abaixo executa o mesmo fluxo do terminal e atualiza o relatório e os gráficos em saida/. As fórmulas de TSP em docs/MODELAGEM-FORMAL.md descrevem o modelo exato de referência; o código utiliza permutações e busca local, sem solver de programação inteira.


In [ ]:
from executar_roteiro import main
main()
